In [21]:
import re
from typing import Any

from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.agents.middleware import (
    AgentMiddleware,
    AgentState,
    HumanInTheLoopMiddleware,
    PIIDetectionError,
    PIIMiddleware,
    hook_config,
)
from langchain_core.messages import AIMessage
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.runtime import Runtime
from langgraph.types import Command

load_dotenv()

AGENT_MODEL = "gpt-4o-mini"

## Shared tools & helpers

In [22]:
@tool
def search_tool(query: str) -> str:
    """Search for information."""
    return f"Results for: {query}"


@tool
def customer_lookup(query: str) -> str:
    """Look up customer information."""
    return f"Customer record found for query: {query}."


@tool
def send_email(to: str, subject: str, body: str) -> str:
    """Send an email to a recipient."""
    return f"Email sent to {to} with subject: {subject} and body: {body}"


@tool
def delete_records(table: str, condition: str) -> str:
    """Delete records from the database."""
    return f"Deleted records from {table} where {condition}"


def ask(agent, text: str, config: dict | None = None) -> str:
    """Send one user message to an agent and return the final reply text."""
    result = agent.invoke({"messages": [{"role": "user", "content": text}]}, config)
    return result["messages"][-1].text

## Deterministic approach

In [23]:
BANNED_KEYWORDS = ["kill", "hack", "exploit", "malware", "bomb"]


def make_keyword_filter(keywords: list[str]):
    """Return find(text) -> first banned keyword found (or None).

    Matches at word starts, so "hacking" is caught but "skill" is not.
    """
    pattern = re.compile(r"\b(?:" + "|".join(map(re.escape, keywords)) + ")", re.IGNORECASE)

    def find(text: str) -> str | None:
        match = pattern.search(text)
        return match.group(0).lower() if match else None

    return find


find_banned = make_keyword_filter(BANNED_KEYWORDS)


def deterministic_guardrail(text: str) -> bool:
    """Returns True if content is blocked."""
    return find_banned(text) is not None


test_input = [
    "How do I hack into Google server?",
    "What is LLM Guardrails?",
    "How do I kill my enemy?",
    "How does malware spread?",
]

print("---- Deterministic Approach ----")
for inp in test_input:
    print(f"{'BLOCKED' if deterministic_guardrail(inp) else 'ALLOWED'}: {inp}")

---- Deterministic Approach ----
BLOCKED: How do I hack into Google server?
ALLOWED: What is LLM Guardrails?
BLOCKED: How do I kill my enemy?
BLOCKED: How does malware spread?


## Model based approach

In [24]:
judge = ChatOpenAI(model=AGENT_MODEL, temperature=0)


def model_based_guardrail(text: str, kind: str = "user input") -> bool:
    """Uses an LLM to evaluate safety. Returns True if UNSAFE."""
    verdict = judge.invoke(
        f"Is the following {kind} safe and appropriate? "
        f"Reply with only 'SAFE' or 'UNSAFE'.\n\n{kind}: {text}"
    )
    return "UNSAFE" in verdict.text.upper()


print("---- Model based approach ----")
for inp in test_input:
    print(f"{'UNSAFE' if model_based_guardrail(inp) else 'SAFE'}: {inp}")

---- Model based approach ----
UNSAFE: How do I hack into Google server?
SAFE: What is LLM Guardrails?
UNSAFE: How do I kill my enemy?
SAFE: How does malware spread?


## PII Detection Middleware

|Strategy | Result |
|--|--| 
| redact | [REDACTED_EMAIL] |
| mask | ********* |
| hash | a8sdf2709sdf |
| block | Raises an exception |

In [25]:
API_KEY_PATTERN = r"sk-[A-Za-z0-9_-]{20,}"  # also matches newer sk-proj-... keys

pii_agent = create_agent(
    model=AGENT_MODEL,
    tools=[customer_lookup],
    middleware=[
        # Redact emails in user input before sending to model
        PIIMiddleware("email", strategy="redact"),
        # Mask credit cards in user input
        PIIMiddleware("credit_card", strategy="mask"),
        # Block API keys - raise error if detected
        PIIMiddleware("api_key", detector=API_KEY_PATTERN, strategy="block"),
    ],
)

print("Agent with PII middleware created successfully!")

Agent with PII middleware created successfully!


In [26]:
result = pii_agent.invoke({
    "messages": [{
        "role": "user",
        "content": "My email is john.doe@gmail.com and my card is 1230-3422-4892-1237. Can you help me?",
    }]
})

print("Model saw:", result["messages"][0].text)
print("Agent:    ", result["messages"][-1].text)

Model saw: My email is [REDACTED_EMAIL] and my card is ****-****-****-1237. Can you help me?
Agent:     I found your customer record associated with your email. How can I assist you further?


### API Key blocking

In [27]:
try:
    ask(pii_agent, "Here is my key: sk-asdfpoiu12347890asdfpoiu12347890")
except PIIDetectionError as e:
    print(f"Blocked as expected: {e}")

Blocked as expected: Detected 1 instance(s) of api_key in text content


## Human in the loop

In [28]:
# Only tools listed in interrupt_on need approval; all others run automatically.
hitl_agent = create_agent(
    model=AGENT_MODEL,
    tools=[search_tool, send_email, delete_records],
    middleware=[
        HumanInTheLoopMiddleware(interrupt_on={"send_email": True, "delete_records": True}),
    ],
    checkpointer=InMemorySaver(),
)

print("HITL agent created!")

HITL agent created!


In [29]:
# Step 1: Invoke -- agent pauses before send_email
config = {"configurable": {"thread_id": "session_001"}}
ask(hitl_agent, "Send an email to team@company.com about the Q4 results", config)
print("---- Agent paused -- awaiting human approval ----")

---- Agent paused -- awaiting human approval ----


In [30]:
# Step 2: Human reviews and APPROVES (same thread_id resumes the paused session)
approved = hitl_agent.invoke(
    Command(resume={"decisions": [{"type": "approve"}]}),
    config=config,
)
print("---- Approved ----")
print(approved["messages"][-1].text)

---- Approved ----
The email regarding the Q4 results has been successfully sent to team@company.com. If you need any further assistance, feel free to ask!


In [31]:
# Alternative -- Human REJECTS
config2 = {"configurable": {"thread_id": "session_002"}}
ask(hitl_agent, "Delete all records from the users table where active=false", config2)

rejected = hitl_agent.invoke(
    Command(resume={"decisions": [{"type": "reject", "message": "Too risky, needs DBA review"}]}),
    config=config2,
)
print("---- Rejected ----")
print(rejected["messages"][-1].text)

---- Rejected ----
It seems that the operation to delete records from the users table is considered too risky and requires DBA review. You may want to consult with your database administrator to proceed with this request.


## Custom Before Agent (input filter)

In [32]:
class ContentFilterMiddleware(AgentMiddleware):
    """
    Deterministic guardrail: block requests containing banned keywords.
    Runs BEFORE the agent processes anything -- zero LLM cost for blocked requests.
    """

    def __init__(self, keywords: list[str]):
        super().__init__()
        self.find_banned = make_keyword_filter(keywords)

    @hook_config(can_jump_to=["end"])
    def before_agent(self, state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        # The newest message is the user's current request (works for multi-turn threads too).
        last = state["messages"][-1] if state["messages"] else None
        if last is None or last.type != "human":
            return None

        keyword = self.find_banned(last.text)
        if keyword is None:
            return None

        print(f"Blocked -- keyword detected: '{keyword}'")
        return {
            "messages": [AIMessage("I cannot process requests containing inappropriate content. Please rephrase your request.")],
            "jump_to": "end",
        }


filtered_agent = create_agent(
    model=AGENT_MODEL,
    tools=[search_tool],
    middleware=[ContentFilterMiddleware(BANNED_KEYWORDS)],
)

In [33]:
# Test 1 - Safe request
print("Safe request:", ask(filtered_agent, "What is RAG?"))

# Test 2 - Unsafe request
print("Unsafe request:", ask(filtered_agent, "How do I hack?"))

Safe request: RAG stands for "Red, Amber, Green," and it is a status reporting tool commonly used in project management. The RAG system provides a simple way to assess and communicate the status of a project or specific tasks:

- **Red**: Indicates that there are significant issues or problems that need immediate attention, and the project is at risk of not meeting its objectives.
- **Amber**: Suggests that there are some issues, but they are being managed. The project may still be at risk, but it is not as critical as a red status.
- **Green**: Signifies that the project is on track, with no significant issues or risks present.

This color-coded system helps stakeholders quickly understand project health and make informed decisions based on the overall status.
Blocked -- keyword detected: 'hack'
Unsafe request: I cannot process requests containing inappropriate content. Please rephrase your request.


## Custom After Agent (output filter)

In [34]:
class SafetyGuardrailMiddleware(AgentMiddleware):
    """
    Model-based guardrail: an LLM judges the response before it reaches the user.
    Runs AFTER the agent finishes.
    """

    FALLBACK = "I'm unable to provide that response. Please rephrase your request or contact support."

    def after_agent(self, state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        last = state["messages"][-1] if state["messages"] else None
        # Skip non-AI messages and empty (tool-call-only) messages: saves an LLM call.
        if not isinstance(last, AIMessage) or not last.text:
            return None

        if model_based_guardrail(last.text, kind="AI response"):
            print("Output flagged as UNSAFE -- replacing with safe fallback")
            # Reusing the id makes the reducer replace the message instead of appending.
            return {"messages": [AIMessage(self.FALLBACK, id=last.id)]}
        return None


safe_agent = create_agent(
    model=AGENT_MODEL,
    tools=[search_tool],
    middleware=[SafetyGuardrailMiddleware()],
)

In [35]:
# Test output safety check
print("Response:", ask(safe_agent, "What is the weather like today?"))

Response: I wasn't able to retrieve the current weather information. You can check a weather website or app for the most accurate and up-to-date information.


## Layered Guardrails

In [37]:
production_agent = create_agent(
    model=AGENT_MODEL,
    tools=[search_tool, send_email],
    middleware=[
        # Layer 1: Deterministic input filter (before agent)
        ContentFilterMiddleware(BANNED_KEYWORDS),
        # Layer 2: PII masking on input
        PIIMiddleware("credit_card", strategy="mask"),
        # Layer 3: Human approval for sensitive tools
        HumanInTheLoopMiddleware(interrupt_on={"send_email": True}),
        # Layer 4: PII redaction on output
        PIIMiddleware("email", strategy="redact", apply_to_input=False, apply_to_output=True),
        # Layer 5: Model-based output safety
        SafetyGuardrailMiddleware(),
    ],
    checkpointer=InMemorySaver(),
)

print("Agent with 5-layer guardrails created!")

Agent with 5-layer guardrails created!
